# What Do We Learn From the Past Replays?




## 1. Complete programme with guarded execution

Let $R(t)$ be the full programme action at turn $t$, and let $G$ be the visible-state guard network. The executed policy is

$$
\pi_t = G\!\left(x_t,R(t)\right).
$$

The guards align the live hand count, repair weeds, project same-turn shed operations, cap sales by executable inventory, order contested sales, and fund purchases sequentially. A failed precondition suppresses only the infeasible part and falls back to the programme elsewhere.


## 2. Conserved livestock substitution

At registered pasture-compatible purchase nodes, dairy and wool pressure are

$$
\begin{aligned}
P_{\mathrm{wool}}
    &=2N_{\mathrm{Yarn}}+\frac{p_{\mathrm{wool}}}{200},\\
P_{\mathrm{dairy}}
    &=N_{\mathrm{Pizza}}+N_{\mathrm{IceCream}}
      +N_{\mathrm{Smoothie}}+\frac{p_{\mathrm{milk}}}{160}.
\end{aligned}
$$

If the pressure gap crosses its guard, a complete cow or sheep transaction bundle is relabelled. The transformation preserves

$$
\begin{aligned}
\text{pasture geometry}' &= \text{pasture geometry},\\
\text{animal count}' &= \text{animal count},\\
\text{service route}' &= \text{service route}.
\end{aligned}
$$

Animal-product sale slots are then assigned to the executable item with the greater exact marginal revenue.


## 3. Guarded latent-pasture activation

At the verified checkpoint, an extra animal-hand bundle is activated only when the target is an empty serviced pasture and all visible feasibility conditions hold:

$$
\mathcal{F}
=\mathcal{F}_{\mathrm{geometry}}
\cap\mathcal{F}_{\mathrm{inventory}}
\cap\mathcal{F}_{\mathrm{capacity}}
\cap\mathcal{F}_{\mathrm{cash}}.
$$

The new hand follows a fixed pickup, movement, and placement transaction. Any mismatch cancels the intervention and returns control to the parent programme.


## 4. Exact terminal frontier

Let $T=718$ be the final executable turn. For remaining quantity $q_i$ of product $i$ and current market inventory $I_i$, its executable terminal revenue is

$$
V_i(q_i;I_i)
=\sum_{k=0}^{q_i-1}p_i(I_i+k).
$$

The engine guarantees $p_i(I)\ge1$. Since inventory has no continuation value after $T$,

$$
q_i>0 \quad\Longrightarrow\quad V_i(q_i;I_i)>0.
$$

Therefore final liquidation weakly dominates retention. The frontier first expands an existing sell order for the same product, which costs no additional slot. If only $K$ new order slots remain, products without an existing slot are selected by

$$
\underset{S}{\operatorname{maximize}}
\quad\sum_{i\in S}V_i(q_i;I_i)
\qquad\text{subject to}\qquad |S|\le K.
$$

Because every selected product consumes one slot and revenues are separable, choosing the $K$ largest values is exact.


In [ ]:
# Decode, verify, and write the complete submission package.
import base64
import hashlib
import io
import json
from pathlib import Path
import tarfile

ARCHIVE_B64 = (
    "..."
)
EXPECTED_ARCHIVE_SHA256 = "ecec6aaf203ed1ecf0736bfa3706541d850144590d0757b177140c7faf087eaf"
EXPECTED_MEMBER_HASHES = json.loads('{"agents/complete_terminal_frontier.py": "4bd3d63957d1d74d6efd8d08e3c5d3e4636c0da726d4baf3c691c7c877fd930d", "agents/e749a_niklita_consensus_network.py": "2188debac2af3308f4ea1bd427a38cd3a2332394073c7cca6240c011fb0c5374", "agents/e750a_place_funding_repair.py": "b0e12951d73646f37eb3b67531716237520f67cdea723914a87bda764c712dc0", "agents/e766a_universal_kenjo_medoid.py": "b66a4acb25695ce7e04b4a9a57c60543926fdc3bd2e85fc827c82bc80d86dc45", "agents/e773a_demand_aligned_pasture_network.py": "f7e31def12b377a74a777561ea0f65e571be1e0d735ae4eb1825e70c0c8bafae", "agents/e774a_terminal_animal_frontier.py": "bc446060cb4ca57e31cc7583f0da16e92c31cb0818d84435483f7523282f4b3c", "agents/e775a_latent_pasture_activation.py": "4abb721b60f8928683ce038903f616cc618ce9a547c7a1f520b1fb81a7398bc6", "agents/e776a_engine_exact_latent_pasture.py": "2c603fd6f9bf978d7e8eef78a34647bd379d0e89963ada0cbb1c52c39b0f4b3f", "artifacts/e706_top10_tapes/episode_101408728_seat1.py": "685cdf4d9b14f16985e0dc7bdc0770446c3862a233be5dfe1f0efe508eedb629", "artifacts/e751_current_top10_tapes/episode_102192548_seat1.py": "4d686ff547797f776081254ee602eda151ec0fe5de7ddcdf646fedf1bf5f0f89", "configs/server_environment_20260807.json": "ca5cb5d68d2b3cca65016c6b4eb62d3a5aa1c463720b7c2fe7bc5689a38e5d1a", "e776_pkg/__init__.py": "b001950885c35bcf24367510245f41667e85bad8d754072c3489afdcf97184c4", "e776_pkg/entry.py": "5aecede545c47e7b2ea19228bd1bd9d0083983e65d4f8d01a940a6470a5ef4ad", "main.py": "e8498c67914ecc607ae69fde25a728361eb5acea94c00ecc85deffbdafe50413", "optimized_pkg/__init__.py": "f1e043b916ff50e95d071dc36888ec2a6cda0e9f460d2d34dabbdaa9f033854a", "optimized_pkg/entry.py": "d8e0cb1200f024e88a6103907ca66e829036694b05d1d56ffa5a7b65f247a81a"}')

ARCHIVE_BYTES = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(ARCHIVE_BYTES).hexdigest() == EXPECTED_ARCHIVE_SHA256
with tarfile.open(fileobj=io.BytesIO(ARCHIVE_BYTES), mode="r:gz") as package:
    members = {member.name: package.extractfile(member).read()
               for member in package.getmembers() if member.isfile()}
assert set(members) == set(EXPECTED_MEMBER_HASHES)
for name, data in members.items():
    assert hashlib.sha256(data).hexdigest() == EXPECTED_MEMBER_HASHES[name]

ARCHIVE_PATH = Path.cwd() / "submission.tar.gz"
MAIN_PATH = Path.cwd() / "main.py"
ARCHIVE_PATH.write_bytes(ARCHIVE_BYTES)
MAIN_PATH.write_bytes(members["main.py"])
print("archive:", ARCHIVE_PATH)
print("main.py:", MAIN_PATH)
print("members:", len(members))


In [ ]:
# Extract into an isolated folder and import the packaged entry point.
import importlib.util
import sys

PACKAGE_DIR = Path.cwd() / "generated_submission"
PACKAGE_DIR.mkdir(exist_ok=True)
with tarfile.open(ARCHIVE_PATH, "r:gz") as package:
    package.extractall(PACKAGE_DIR)

sys.path.insert(0, str(PACKAGE_DIR))
spec = importlib.util.spec_from_file_location(
    "generated_submission_main", PACKAGE_DIR / "main.py"
)
submission = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = submission
spec.loader.exec_module(submission)
assert callable(submission.kaggriculture_agent)
print("standalone package import: OK")


## Attribution

The complete programme priors are the attributed CC0 public traces from Kenjo1209, episode 102192548 seat 1, and NIklitaCheporev, episode 101408728 seat 1. The supplied guard, demand-alignment, and latent-pasture network is preserved byte-for-byte; the added contribution is the complete terminal frontier described above.
